# E021 training on Kaggle -- fold 0 (third ensemble member)

CODING_SPEC_05_ENSEMBLE.md: E007's exact recipe with seed 1337 instead of 42 (lowest-risk added member) -- see
configs/E021-f0.yaml.

Setup (manual, one time):
1. Re-package with `scripts/package_kaggle.py` (so configs/E021-f*.yaml are
   in it), push a new version of `potcharapanchumrak/solar-k-package`, attach it and the
   `filament-segmentation-2026` competition.
2. Turn on a GPU accelerator and Internet access (pip + timm ImageNet weights).

Checkpoints/logs/probs write to `/kaggle/working/outputs`.

In [ ]:
import os
import time

PKG_DIR = '/kaggle/input/datasets/potcharapanchumrak/solar-k-package'

for attempt in range(10):
    if os.path.isdir(PKG_DIR):
        break
    print(f'{PKG_DIR} not mounted yet, waiting (attempt {attempt + 1}/10)...')
    time.sleep(3)
else:
    raise FileNotFoundError(f'{PKG_DIR} never appeared after 30s -- check the dataset is attached')

os.chdir(PKG_DIR)
print(os.listdir('.'))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os

COMP_ROOT = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
PKG_SPLITS = os.path.join(PKG_DIR, 'data', 'splits')
DATA_LINK_ROOT = '/kaggle/working/data_root'
os.makedirs(DATA_LINK_ROOT, exist_ok=True)
for name, target in [('raw', COMP_ROOT), ('splits', PKG_SPLITS)]:
    link = os.path.join(DATA_LINK_ROOT, name)
    if not os.path.islink(link):
        os.symlink(target, link)

os.environ['SOLAR_K_DATA'] = DATA_LINK_ROOT
os.environ['SOLAR_K_OUTPUT'] = '/kaggle/working/outputs'
os.environ['SOLAR_K_CACHE'] = '/kaggle/temp/solar_k_cache'

RESUME_FROM = None
MAX_HOURS = 8.5

In [ ]:
import subprocess

cmd = ['python', 'scripts/train.py', '--config', 'configs/E021-f0.yaml', '--max_hours', str(MAX_HOURS)]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]

# E007 took ~46 min/fold on a 4090; estimated ~2.5-3 h/fold on a T4,
# well under MAX_HOURS; train.py checkpoints every 1000 iters, so a session
# that dies can resume via RESUME_FROM.
subprocess.run(cmd, check=True)